In [7]:
# ---- CELL 1: Imports & Setup ----
import torch
import numpy as np
from PIL import Image
from diffusers import StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)


Running on: cuda


In [8]:
# ---- CELL 2: Load CLIP & Stable Diffusion ----
clip_model = CLIPModel.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")

sd_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
sd_pipe = sd_pipe.to(device)


KeyboardInterrupt: 

In [ ]:
# ---- CELL 2: Load CLIP & Stable Diffusion ----
clip_model = CLIPModel.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("laion/CLIP-ViT-H-14-laion2B-s32B-b79K")
# ---- CELL 3: Load your two images ----
img_path1 = "../images/japan/DSC00413.jpg"   # <--- replace with your actual filename
img_path2 = "../images/japan/DSC00448.jpg"  # <--- replace with your actual filename

img1 = Image.open(img_path1).convert("RGB")
img2 = Image.open(img_path2).convert("RGB")
# ---- CELL 4: Encode images to CLIP vectors ----
def img_to_clip_vec(img):
    inputs = clip_processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        vec = clip_model.get_image_features(**inputs).cpu().numpy().flatten()
    return vec

vec1 = img_to_clip_vec(img1)
vec2 = img_to_clip_vec(img2)
print("vec1 shape:", vec1.shape)
print("vec2 shape:", vec2.shape)


vec1 shape: (1024,)
vec2 shape: (1024,)


# UNCLIP

In [ ]:
# ---- CELL 1: Setup and imports ----
import torch
from PIL import Image
import numpy as np
from diffusers import StableUnCLIPImg2ImgPipeline
from transformers import CLIPProcessor, CLIPModel
import os

# ---- DEVICE ----
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ---- CELL 2: Load OpenCLIP ViT-H-14 for SD2.1-unCLIP ----
print("Loading CLIP ViT-H-14 model and processor...")
clip_model = CLIPModel.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
).to(device).eval()

clip_processor = CLIPProcessor.from_pretrained(
    "laion/CLIP-ViT-H-14-laion2B-s32B-b79K"
)
print("CLIP model and processor loaded.")

# ---- CELL 3: Load your images ----
img_path1 = "../images/japan/DSC00544.jpg"  # <- update as needed
img_path2 = "../images/japan/DSC00448.jpg"    # <- update as needed

print(f"Loading images:\n  1: {img_path1}\n  2: {img_path2}")
img1 = Image.open(img_path1).convert("RGB")
img2 = Image.open(img_path2).convert("RGB")

# Preprocess for OpenCLIP ViT-H-14
inputs1 = clip_processor(images=img1, return_tensors="pt").to(device)
inputs2 = clip_processor(images=img2, return_tensors="pt").to(device)

# ---- CELL 4: Encode images to CLIP ViT-H-14 vectors ----
print("Encoding images to CLIP vectors...")
with torch.no_grad():
    vec1 = clip_model.get_image_features(**inputs1)
    vec2 = clip_model.get_image_features(**inputs2)
    # Normalize (important for SD2.1-unCLIP)
    vec1 = vec1 / vec1.norm(dim=-1, keepdim=True)
    vec2 = vec2 / vec2.norm(dim=-1, keepdim=True)
print("vec1 shape:", tuple(vec1.shape), "| vec2 shape:", tuple(vec2.shape))

# ---- CELL 5: Interpolate between vectors ----
def interpolate(v1, v2, n_steps):
    return [v1 + (v2 - v1) * (i / (n_steps + 1)) for i in range(1, n_steps + 1)]

n_interp = 4  # how many frames between
vecs = [vec1] + interpolate(vec1, vec2, n_interp) + [vec2]
print(f"Prepared {len(vecs)} CLIP vectors for image generation (including endpoints).")

# ---- CELL 6: Load Stable unCLIP pipeline ----
print("Loading Stable unCLIP (SD 2.1) pipeline...")
pipe = StableUnCLIPImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-2-1-unclip",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)
print("Pipeline loaded.")

# ---- CELL 7: Generate and save images from interpolated vectors ----
out_dir = "unclip_interp"
os.makedirs(out_dir, exist_ok=True)
print(f"Generating and saving images to '{out_dir}/'...")




Using device: cuda
Loading CLIP ViT-H-14 model and processor...
CLIP model and processor loaded.
Loading images:
  1: ../images/japan/DSC00544.jpg
  2: ../images/japan/DSC00448.jpg
Encoding images to CLIP vectors...
vec1 shape: (1, 1024) | vec2 shape: (1, 1024)
Prepared 6 CLIP vectors for image generation (including endpoints).
Loading Stable unCLIP (SD 2.1) pipeline...


Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Pipeline loaded.
Generating and saving images to 'unclip_interp/'...


In [ ]:
# ---- CELL 7: Generate and save images from original embeddings only ----
out_dir = "unclip_recon"
os.makedirs(out_dir, exist_ok=True)
print(f"Generating and saving reconstructed images to '{out_dir}/'...")

clip_vecs = [vec1, vec2]
img_names = [os.path.basename(img_path1), os.path.basename(img_path2)]

for idx, (v, name) in enumerate(zip(clip_vecs, img_names)):
    with torch.no_grad():
        # Convert to float16 if needed
        v_fp16 = v.half() if pipe.dtype == torch.float16 else v
        image = pipe(
            image_embeds=v_fp16,
            num_inference_steps=40,
            guidance_scale=10
        ).images[0]
    out_path = os.path.join(out_dir, f"recon_{idx+1}_{name}")
    image.save(out_path)
    print(f"Saved reconstructed image: {out_path}")

print("Done! Compare outputs to your originals.")


Generating and saving reconstructed images to 'unclip_recon/'...


  0%|          | 0/41 [00:00<?, ?it/s]

Saved reconstructed image: unclip_recon\recon_1_DSC00544.jpg


  0%|          | 0/41 [00:00<?, ?it/s]

Saved reconstructed image: unclip_recon\recon_2_DSC00448.jpg
Done! Compare outputs to your originals.


In [ ]:
for idx, v in enumerate(vecs):
    with torch.no_grad():
        # v must be shape (1, 1024)
        image = pipe(
            image_embeds=v.half(),
            num_inference_steps=40,
            guidance_scale=7.5
        ).images[0]

    out_path = os.path.join(out_dir, f"frame_{idx:02d}.png")
    image.save(out_path)
    print(f"Saved {out_path}")

print("All frames generated and saved.")

  0%|          | 0/41 [00:00<?, ?it/s]

Saved unclip_interp\frame_00.png


  0%|          | 0/41 [00:00<?, ?it/s]

Saved unclip_interp\frame_01.png


  0%|          | 0/41 [00:00<?, ?it/s]

...

Saved unclip_interp\frame_03.png


  0%|          | 0/41 [00:00<?, ?it/s]

Saved unclip_interp\frame_04.png


  0%|          | 0/41 [00:00<?, ?it/s]

Saved unclip_interp\frame_05.png
All frames generated and saved.


# SDXL turbo

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load your image and preserve aspect ratio
img_path = "../images/japan/DSC00821.jpg"
init_image = Image.open(img_path).convert("RGB")

# Optional: Resize so the largest side is 1024 (recommended for SDXL Turbo)
max_side = 1024
w, h = init_image.size
if max(w, h) > max_side:
    scale = max_side / max(w, h)
    new_w, new_h = int(w * scale), int(h * scale)
    # Make sure dimensions are multiples of 8 (required by SDXL)
    new_w, new_h = (new_w // 8) * 8, (new_h // 8) * 8
    init_image = init_image.resize((new_w, new_h), Image.LANCZOS)
else:
    # Round to multiples of 8
    new_w, new_h = (w // 8) * 8, (h // 8) * 8
    init_image = init_image.resize((new_w, new_h), Image.LANCZOS)

# --- Load SDXL Turbo img2img pipeline ---
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

import time

start_time = time.time()
# --- Define your prompt and settings ---
prompt = "lost transients of astral bodies"
negative_prompt = "text, letters, watermark, logo, blurry, low quality"
strength = 0.7
generator = torch.manual_seed(2222)

# --- Run the pipeline ---
output = pipe(
    prompt=prompt,
    image=init_image,
    negative_prompt=negative_prompt,
    strength=strength,
    num_inference_steps=4,  # SDXL Turbo is designed for few steps (e.g., 1-4)
    guidance_scale=1.0,     # Lower values for Turbo
    generator=generator
).images[0]

end_time = time.time()
print(f"Image generation took {end_time - start_time:.2f} seconds")
output.save("output_sdxl_turbo_img2img.png")
output.show()
print("Saved output as output_sdxl_turbo_img2img_2.png")



Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Image generation took 3.89 seconds
Saved output as output_sdxl_turbo_img2img_2.png


In [1]:
import os
import random
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image

# Settings
image_folder = "../images/japan/"
output_folder = "img2img_outputs"
os.makedirs(output_folder, exist_ok=True)
max_side = 1024
num_samples = 10

# 10 prompts (edit these as needed)
prompts = [
    "lost transients of astral bodies",
    "walking in the forest of the mind",
    "the temple dissolves into morning fog",
    "a dragonfly circling a sacred pond",
    "dreams of cobalt mountains",
    "a lantern in endless rain",
    "moonlit path to forgotten shrines",
    "echoes of silent laughter",
    "clouds tangled in old wires",
    "bamboo wind and digital ghosts"
]

negative_prompt = "text, letters, watermark, logo, blurry, low quality"
strength = 0.7
num_inference_steps = 4
guidance_scale = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

pipe.enable_vae_slicing()
pipe.safety_checker = None

# Get random images from the folder
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
selected_files = random.sample(image_files, num_samples)

for idx, (img_name, prompt) in enumerate(zip(selected_files, prompts)):
    img_path = os.path.join(image_folder, img_name)
    init_image = Image.open(img_path).convert("RGB")

    # Resize: preserve aspect and force multiples of 8
    w, h = init_image.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        new_w, new_h = (new_w // 8) * 8, (new_h // 8) * 8
    else:
        new_w, new_h = (w // 8) * 8, (h // 8) * 8
    init_image = init_image.resize((new_w, new_h), Image.LANCZOS)

    # Timing
    import time
    
    generator = torch.manual_seed(2222 + idx)  # Different seed per image for more diversity
    start_time = time.time()
    
    output = pipe(
        prompt=prompt,
        image=init_image,
        negative_prompt=negative_prompt,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        
    ).images[0]
    end_time = time.time()
    
    # Save original image
    orig_out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_original.png"
    orig_out_path = os.path.join(output_folder, orig_out_name)
    init_image.save(orig_out_path)

    # Sanitize prompt for filename (remove problematic characters)
    safe_prompt = "".join(c if c.isalnum() or c in " _-" else "_" for c in prompt)[:60]

    # Save generated image with prompt in filename
    out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_{safe_prompt}_img2img.png"
    out_path = os.path.join(output_folder, out_name)
    output.save(out_path)
    print(f"[{idx+1}/10] {img_name} | Prompt: \"{prompt}\" | Size: {new_w}x{new_h} | Time: {end_time - start_time:.2f}s | Saved: {out_path}")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[1/10] DSC08303.jpg | Prompt: "lost transients of astral bodies" | Size: 1024x680 | Time: 1.91s | Saved: img2img_outputs\00_DSC08303_lost transients of astral bodies_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

[2/10] DSC06714.jpg | Prompt: "walking in the forest of the mind" | Size: 1024x680 | Time: 1.02s | Saved: img2img_outputs\01_DSC06714_walking in the forest of the mind_img2img.png


...

[8/10] DSC07938.jpg | Prompt: "echoes of silent laughter" | Size: 1024x680 | Time: 1.06s | Saved: img2img_outputs\07_DSC07938_echoes of silent laughter_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

[9/10] DSC07193.jpg | Prompt: "clouds tangled in old wires" | Size: 1024x680 | Time: 0.97s | Saved: img2img_outputs\08_DSC07193_clouds tangled in old wires_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

[10/10] DSC07243.jpg | Prompt: "bamboo wind and digital ghosts" | Size: 1024x680 | Time: 1.04s | Saved: img2img_outputs\09_DSC07243_bamboo wind and digital ghosts_img2img.png


In [1]:
import os
import random
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image

# Settings
image_folder = "../images/japan/"
output_folder = "img2img_outputs"
os.makedirs(output_folder, exist_ok=True)
max_side = 1024
num_samples = 10

# 10 prompts (edit these as needed)
prompts = [
    "lost transients of astral bodies",
    "walking in the forest of the mind",
    "the temple dissolves into morning fog",
    "a dragonfly circling a sacred pond",
    "dreams of cobalt mountains",
    "a lantern in endless rain",
    "moonlit path to forgotten shrines",
    "echoes of silent laughter",
    "clouds tangled in old wires",
    "bamboo wind and digital ghosts"
]

negative_prompt = "text, letters, watermark, logo, blurry, low quality"
strength = 0.7
num_inference_steps = 4
guidance_scale = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

pipe.enable_vae_slicing()
pipe.safety_checker = None

# Get random images from the folder
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
selected_files = random.sample(image_files, num_samples)

for idx, (img_name, prompt) in enumerate(zip(selected_files, prompts)):
    img_path = os.path.join(image_folder, img_name)
    init_image = Image.open(img_path).convert("RGB")

    # Resize: preserve aspect and force multiples of 8
    w, h = init_image.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        new_w, new_h = (new_w // 8) * 8, (new_h // 8) * 8
    else:
        new_w, new_h = (w // 8) * 8, (h // 8) * 8
    init_image = init_image.resize((new_w, new_h), Image.LANCZOS)

    # Timing
    import time
    
    generator = torch.manual_seed(2222 + idx)  # Different seed per image for more diversity
    start_time = time.time()
    
    output = pipe(
        prompt=prompt,
        image=init_image,
        negative_prompt=negative_prompt,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
        
    ).images[0]
    end_time = time.time()
    
    # Save original image
    orig_out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_original.png"
    orig_out_path = os.path.join(output_folder, orig_out_name)
    init_image.save(orig_out_path)

    # Sanitize prompt for filename (remove problematic characters)
    safe_prompt = "".join(c if c.isalnum() or c in " _-" else "_" for c in prompt)[:60]

    # Save generated image with prompt in filename
    out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_{safe_prompt}_img2img.png"
    out_path = os.path.join(output_folder, out_name)
    output.save(out_path)
    print(f"[{idx+1}/10] {img_name} | Prompt: \"{prompt}\" | Size: {new_w}x{new_h} | Time: {end_time - start_time:.2f}s | Saved: {out_path}")


A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "c:\Users\skite\miniconda3\envs\goofi-pipe\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

[1/10] DSC07934.jpg | Prompt: "lost transients of astral bodies" | Size: 1024x680 | Time: 3.55s | Saved: img2img_outputs\00_DSC07934_lost transients of astral bodies_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

...

[8/10] DSC03650.jpg | Prompt: "echoes of silent laughter" | Size: 1024x680 | Time: 1.54s | Saved: img2img_outputs\07_DSC03650_echoes of silent laughter_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

[9/10] DSC08855.jpg | Prompt: "clouds tangled in old wires" | Size: 1024x680 | Time: 0.85s | Saved: img2img_outputs\08_DSC08855_clouds tangled in old wires_img2img.png


  0%|          | 0/2 [00:00<?, ?it/s]

[10/10] DSC01060.jpg | Prompt: "bamboo wind and digital ghosts" | Size: 1024x680 | Time: 0.87s | Saved: img2img_outputs\09_DSC01060_bamboo wind and digital ghosts_img2img.png


In [ ]:
import os
import random
import torch
from diffusers import FluxImg2ImgPipeline
from PIL import Image
import time
import gc
import psutil

# Settings
image_folder = "../images/japan/"
output_folder = "img2img_outputs_flux"
os.makedirs(output_folder, exist_ok=True)
max_side = 1024
num_samples = 10

gc.collect()
torch.cuda.empty_cache()


# Prompts
prompts = [
    "lost transients of astral bodies",
    "walking in the forest of the mind",
    "the temple dissolves into morning fog",
    "a dragonfly circling a sacred pond",
    "dreams of cobalt mountains",
    "a lantern in endless rain",
    "moonlit path to forgotten shrines",
    "echoes of silent laughter",
    "clouds tangled in old wires",
    "bamboo wind and digital ghosts"
]

negative_prompt = "text, letters, watermark, logo, blurry, low quality"
strength = 0.7
num_inference_steps = 4
guidance_scale = 1.0

device = "cuda" if torch.cuda.is_available() else "cpu"
print("loading FLUX.1-schnell pipeline... on ", device)

print("Before:", psutil.virtual_memory())

pipe = FluxImg2ImgPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.float16,  # use float16
    device_map="balanced"
)

print("After:", psutil.virtual_memory())

print("pipeline loaded. Preparing for image generation...")
#pipe.enable_model_cpu_offload()
pipe.safety_checker = None


# Get random images from the folder
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))]
selected_files = random.sample(image_files, num_samples)

for idx, (img_name, prompt) in enumerate(zip(selected_files, prompts)):
    img_path = os.path.join(image_folder, img_name)
    init_image = Image.open(img_path).convert("RGB")

    # Resize: preserve aspect and force multiples of 8
    w, h = init_image.size
    if max(w, h) > max_side:
        scale = max_side / max(w, h)
        new_w, new_h = int(w * scale), int(h * scale)
        new_w, new_h = (new_w // 8) * 8, (new_h // 8) * 8
    else:
        new_w, new_h = (w // 8) * 8, (h // 8) * 8

    print(f"Resizing {img_name} from {w}x{h} to {new_w}x{new_h}")
    init_image = init_image.resize((new_w, new_h), Image.LANCZOS)

    generator = torch.manual_seed(2222 + idx)
    start_time = time.time()

    output = pipe(
        prompt=prompt,
        image=init_image,
        negative_prompt=negative_prompt,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    ).images[0]
    end_time = time.time()

    # Save original image
    orig_out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_original.png"
    orig_out_path = os.path.join(output_folder, orig_out_name)
    init_image.save(orig_out_path)

    # Sanitize prompt for filename
    safe_prompt = "".join(c if c.isalnum() or c in " _-" else "_" for c in prompt)[:60]

    # Save generated image
    out_name = f"{idx:02d}_{os.path.splitext(img_name)[0]}_{safe_prompt}_img2img.png"
    out_path = os.path.join(output_folder, out_name)
    output.save(out_path)
    print(f"[{idx+1}/10] {img_name} | Prompt: \"{prompt}\" | Size: {new_w}x{new_h} | Time: {end_time - start_time:.2f}s | Saved: {out_path}")


A matching Triton is not available, some optimizations will not be enabled
Traceback (most recent call last):
  File "c:\Users\skite\miniconda3\envs\goofi-pipe\lib\site-packages\xformers\__init__.py", line 57, in _is_triton_available
    import triton  # noqa
ModuleNotFoundError: No module named 'triton'


loading FLUX.1-schnell pipeline... on  cuda
Before: svmem(total=33844576256, available=16236634112, percent=52.0, used=17607942144, free=16236634112)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

After: svmem(total=33844576256, available=869228544, percent=97.4, used=32975347712, free=869228544)
pipeline loaded. Preparing for image generation...
Resizing DSC02969.jpg from 6000x4000 to 1024x680


  0%|          | 0/3 [00:00<?, ?it/s]

In [2]:
"""
Batch img2img with Stable Diffusion Turbo
----------------------------------------
• Takes 10 random images from `image_folder`
• Uses 10 matching prompts (edit as you like)
• Saves originals + generated results in `output_folder`
• Fast settings: xFormers, VAE slicing, no safety checker
"""

import os, random, time, torch
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

# ----------------------- CONFIG -----------------------
image_folder   = "../images/japan/"
output_folder  = "turbo_img2img_outputs"
os.makedirs(output_folder, exist_ok=True)

num_samples    = 10          # images/prompts to run
max_side       = 512         # sd-turbo is trained at 512×512
strength       = 0.7
num_steps      = 4           # sd-turbo works best at 1-4 steps
guidance_scale = 1.0         # keep 1.0 (no CFG double-pass)

prompts = [
    "a staircase spiraling into a cloud of clocks",
    "umbrellas blooming in a desert of mirrors",
    "a violin melting over a city skyline at dusk",
    "giant goldfish swimming through a library",
    "a train emerging from a teacup",
    "floating islands tethered by red balloons",
    "a forest of hands reaching for falling stars",
    "a cathedral made of glass feathers",
    "bicycles riding across the surface of the moon",
    "a river of ink flowing through a field of eyes",
]
negative_prompt = "text, letters, watermark, logo, blurry, low quality"

# ----------------------- PIPELINE ---------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stabilityai/sd-turbo",          # *** NON-XL model ***
    torch_dtype=torch.float16,
    variant="fp16",
).to(device)

# speed toggles
pipe.enable_xformers_memory_efficient_attention()  # needs xformers installed
pipe.enable_vae_slicing()
pipe.safety_checker = None        # skip extra CLIP pass
pipe.watermark      = None        # skip watermark

# ---------------------- DATA PICK ---------------------
imgs = [f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))]
selected_files = random.sample(imgs, num_samples)

# --------------------- MAIN LOOP ----------------------
for idx, (fname, prompt) in enumerate(zip(selected_files, prompts)):
    init_path = os.path.join(image_folder, fname)
    init_img  = Image.open(init_path).convert("RGB")

    # --- resize (keep aspect, multiple of 8) ---
    w, h = init_img.size
    if max(w, h) > max_side:
        scale      = max_side / max(w, h)
        new_w      = (int(w * scale) // 8) * 8
        new_h      = (int(h * scale) // 8) * 8
    else:
        new_w, new_h = (w // 8) * 8, (h // 8) * 8
    init_img = init_img.resize((new_w, new_h), Image.LANCZOS)

    # --- generate ---
    gen     = torch.manual_seed(2222 + idx)
    t0      = time.time()
    out_img = pipe(
        prompt               = prompt,
        negative_prompt      = negative_prompt,
        image                = init_img,
        strength             = strength,
        num_inference_steps  = num_steps,
        guidance_scale       = guidance_scale,
        generator            = gen,
    ).images[0]
    t_total = time.time() - t0

    # --- save originals & outputs ---
    safe_prompt = "".join(c if c.isalnum() or c in " _-" else "_" for c in prompt)[:60]
    orig_name   = f"{idx:02d}_{os.path.splitext(fname)[0]}_orig.png"
    gen_name    = f"{idx:02d}_{os.path.splitext(fname)[0]}_{safe_prompt}_turbo.png"

    init_img.save(os.path.join(output_folder, orig_name))
    out_img.save(os.path.join(output_folder, gen_name))

    print(f"[{idx+1:2}/10] {fname} | “{prompt}” | {new_w}×{new_h} | {t_total:.2f}s")

print("Done.")


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_img2img.StableDiffusionImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/2 [00:00<?, ?it/s]

[ 1/10] DSC04204.jpg | “a staircase spiraling into a cloud of clocks” | 512×336 | 0.68s


  0%|          | 0/2 [00:00<?, ?it/s]

...

[ 8/10] DSC02538.jpg | “a cathedral made of glass feathers” | 336×512 | 0.37s


  0%|          | 0/2 [00:00<?, ?it/s]

[ 9/10] DSC05235.jpg | “bicycles riding across the surface of the moon” | 512×336 | 0.32s


  0%|          | 0/2 [00:00<?, ?it/s]

[10/10] DSC04172.jpg | “a river of ink flowing through a field of eyes” | 512×336 | 0.33s
Done.


# SDXL

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load your base image ----
img_path = "../images/japan/DSC00448.jpg"  # Update as needed
init_image = Image.open(img_path).convert("RGB").resize((1024, 1024))  # SDXL likes square images

# ---- Load the SDXL image-to-image pipeline ----
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-refiner-1.0",  # High-quality refiner
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

# ---- Define your prompt and strength ----
prompt = "walking in the forest of the mind"
negative_prompt = "text, letters, watermark, logo, blurry, low quality"
strength = 0.6  # Lower = more faithful to input, higher = more prompt-driven

# ---- Run the pipeline ----
generator = torch.manual_seed(2222)
output = pipe(
    prompt=prompt,
    image=init_image,
    negative_prompt=negative_prompt,
    strength=strength,
    num_inference_steps=40,
    guidance_scale=7.0,
    generator=generator
).images[0]

output.save("output_sdxl_img2img.png")
output.show()
print("Saved output as output_sdxl_img2img.png")


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

Saved output as output_sdxl_img2img.png


In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---- Load your base image (preserve aspect ratio) ----
img_path = "../images/japan/DSC00448.jpg"
init_image = Image.open(img_path).convert("RGB")
orig_size = init_image.size
max_side = 1024
ratio = orig_size[0] / orig_size[1]
if orig_size[0] >= orig_size[1]:
    new_w = max_side
    new_h = int(max_side / ratio)
else:
    new_h = max_side
    new_w = int(max_side * ratio)
resized_image = init_image.resize((new_w, new_h), resample=Image.LANCZOS)

# ---- Load the SDXL Base pipeline ----
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

prompt1 = "walking towards destiny"
prompt2 = "finding hope in the forest"
negative_prompt = "text, letters, watermark, logo, blurry, low quality"
n_steps = 5

# ---- Encode prompts and negatives using encode_prompt ----
def get_full_embeds(pipe, prompt):
    # Returns (prompt_embeds, pooled_prompt_embeds)
    return pipe.encode_prompt(prompt)

# ---- Encode prompts and negatives using encode_prompt ----
def get_full_embeds(pipe, prompt):
    prompt_embeds, _, pooled_prompt_embeds, _ = pipe.encode_prompt(prompt)
    return prompt_embeds, pooled_prompt_embeds

def get_full_negative_embeds(pipe, prompt):
    _, negative_prompt_embeds, _, negative_pooled_prompt_embeds = pipe.encode_prompt(prompt)
    return negative_prompt_embeds, negative_pooled_prompt_embeds

emb1, pooled1 = get_full_embeds(pipe, prompt1)
emb2, pooled2 = get_full_embeds(pipe, prompt2)
neg_embeds, neg_pooled = get_full_negative_embeds(pipe, negative_prompt)



def interpolate_embeddings(e1, e2, n):
    # interpolates both prompt_embeds and pooled_prompt_embeds
    out1 = [e1[0] + (e2[0] - e1[0]) * (i / (n + 1)) for i in range(1, n + 1)]
    out2 = [e1[1] + (e2[1] - e1[1]) * (i / (n + 1)) for i in range(1, n + 1)]
    return list(zip(out1, out2))

embeds = [(emb1, pooled1)] + interpolate_embeddings((emb1, pooled1), (emb2, pooled2), n_steps) + [(emb2, pooled2)]
print(f"Interpolating between prompts over {len(embeds)} steps (including endpoints).")

# ---- Generate images for each embedding ----
out_dir = "sdxl_prompt_interp"
os.makedirs(out_dir, exist_ok=True)

strength = 0.5
generator = torch.manual_seed(2222)

for i, (prompt_emb, pooled_emb) in enumerate(embeds):
    with torch.no_grad():
        result = pipe(
            prompt_embeds=prompt_emb,
            pooled_prompt_embeds=pooled_emb,
            negative_prompt_embeds=neg_embeds,
            negative_pooled_prompt_embeds=neg_pooled,
            image=resized_image,
            strength=strength,
            num_inference_steps=40,
            guidance_scale=7.0,
            generator=generator
        ).images[0]
        out_img = result.resize(orig_size, resample=Image.LANCZOS)
        out_path = os.path.join(out_dir, f"interp_{i:02d}.png")
        out_img.save(out_path)
        print(f"Saved: {out_path}")

print("All prompt-interpolated images generated!")


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Interpolating between prompts over 7 steps (including endpoints).


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: sdxl_prompt_interp\interp_00.png


  0%|          | 0/28 [00:00<?, ?it/s]

...

Saved: sdxl_prompt_interp\interp_04.png


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: sdxl_prompt_interp\interp_05.png


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: sdxl_prompt_interp\interp_06.png
All prompt-interpolated images generated!


## Image Mixing

In [ ]:
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline
from PIL import Image
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

img1_path = "../images/japan/DSC00448.jpg"
img2_path = "../images/japan/DSC00413.jpg"
out_dir = "img_pixel_morph"
os.makedirs(out_dir, exist_ok=True)
latent_size = 1024

def preprocess_img(img_path, size=latent_size):
    img = Image.open(img_path).convert("RGB").resize((size, size), Image.LANCZOS)
    img = np.array(img).astype(np.float32) / 255.0
    return img

img1 = preprocess_img(img1_path)
img2 = preprocess_img(img2_path)

n_steps = 4
imgs = []
for t in np.linspace(0, 1, n_steps):
    img = img1 * (1 - t) + img2 * t
    img = np.clip(img, 0, 1)
    img_pil = Image.fromarray((img * 255).astype(np.uint8))
    imgs.append(img_pil)

print("Loading SDXL pipeline...")
pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16"
).to(device)

prompt = "a harmonious fusion of two scenes, cinematic, photorealistic, surrealist style"
strength = 0.5
guidance_scale = 6.0
num_inference_steps = 20

print("Morphing images and running img2img...")
for i, mix_img in enumerate(imgs):
    mix_img.save(f"{out_dir}/pixel_interp_{i:02d}.png")
    print(f"Saved pixel morph: {out_dir}/pixel_interp_{i:02d}.png")
    out = pipe(
        prompt=prompt,
        image=mix_img,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
    ).images[0]
    out.save(f"{out_dir}/img2img_interp_{i:02d}.png")
    print(f"Saved: {out_dir}/img2img_interp_{i:02d}.png")

print("All pixel morphs and img2img results generated and saved in:", out_dir)


Loading SDXL pipeline...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Morphing images and running img2img...
Saved pixel morph: img_pixel_morph/pixel_interp_00.png


  0%|          | 0/10 [00:00<?, ?it/s]

Saved: img_pixel_morph/img2img_interp_00.png
Saved pixel morph: img_pixel_morph/pixel_interp_01.png


...

Saved: img_pixel_morph/img2img_interp_01.png
Saved pixel morph: img_pixel_morph/pixel_interp_02.png


  0%|          | 0/10 [00:00<?, ?it/s]

Saved: img_pixel_morph/img2img_interp_02.png
Saved pixel morph: img_pixel_morph/pixel_interp_03.png


  0%|          | 0/10 [00:00<?, ?it/s]

Saved: img_pixel_morph/img2img_interp_03.png
All pixel morphs and img2img results generated and saved in: img_pixel_morph


# stream diffusion

In [ ]:
import torch
from streamdiffusion import StableDiffusionImg2ImgPipeline
from PIL import Image
import numpy as np
import os

device = "cuda" if torch.cuda.is_available() else "cpu"

img1_path = "../images/japan/DSC00448.jpg"
img2_path = "../images/japan/DSC00413.jpg"
out_dir = "img_pixel_morph"
os.makedirs(out_dir, exist_ok=True)
latent_size = 1024

def preprocess_img(img_path, size=latent_size):
    img = Image.open(img_path).convert("RGB").resize((size, size), Image.LANCZOS)
    img = np.array(img).astype(np.float32) / 255.0
    return img

img1 = preprocess_img(img1_path)
img2 = preprocess_img(img2_path)

n_steps = 8
imgs = []
for t in np.linspace(0, 1, n_steps):
    img = img1 * (1 - t) + img2 * t
    img = np.clip(img, 0, 1)
    img_pil = Image.fromarray((img * 255).astype(np.uint8))
    imgs.append(img_pil)

print("Loading StreamDiffusion SDXL Img2Img pipeline...")
pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16
).to(device)

prompt = "a harmonious fusion of two scenes, cinematic, photorealistic"
strength = 0.7
guidance_scale = 6.0
num_inference_steps = 40

print("Morphing images and running img2img...")
for i, mix_img in enumerate(imgs):
    mix_img.save(f"{out_dir}/pixel_interp_{i:02d}.png")
    print(f"Saved pixel morph: {out_dir}/pixel_interp_{i:02d}.png")
    out = pipe(
        prompt=prompt,
        image=mix_img,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
    ).images[0]
    out.save(f"{out_dir}/img2img_interp_{i:02d}.png")
    print(f"Saved: {out_dir}/img2img_interp_{i:02d}.png")

print("All pixel morphs and img2img results generated and saved in:", out_dir)


ImportError: cannot import name 'StableDiffusionImg2ImgPipeline' from 'streamdiffusion' (c:\Users\skite\miniconda3\envs\goofi-pipe\lib\site-packages\streamdiffusion\__init__.py)